##  Optimización del Corpus y Generación de Categorías Universales (K=15)

Los objetivos de este noteboook son:implementar un filtro de **Deduplicación Semántica** para limpiar el ruido estructural de la AEMPS;
agrupar la información en **15 clústeres** para maximizar la visibilidad de temas de seguridad (conducción, embarazo, urgencias); y aplicar una capa de **Abstracción Semántica Post-Clustering** mediante un comité de LLMs para transformar etiquetas técnicas en preguntas universales para el paciente.

In [49]:
import pandas as pd
df_parrafos = pd.read_csv("prospectos_limpio_segmentado.csv")

In [47]:
df_parrafos = df_parrafos.drop_duplicates(subset=['parrafo_anonimizado'])

### Mitigación de la Saturación Semántica: Deduplicación al 95%

Tras las fases anteriores, se identificó que el aumento de prospectos (500 documentos) no incrementaba la diversidad de tópicos, sino que reforzaba la redundancia de instrucciones legales estándar (ej: "olvido de dosis").

**Metodología de Limpieza:**
* **Herramienta:** Vectores **TF-IDF** por su alta sensibilidad a la literalidad léxica.
* **Umbral Crítico (0.95):** Se establece tras un análisis de sensibilidad. Un umbral de 0.86 provocaba colisiones entre instrucciones distintas (toma vs. uso), mientras que el 0.95 permite eliminar párrafos con variaciones mínimas de redacción (cuasi-duplicados) sin comprometer la integridad de la información específica de cada fármaco.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def analizar_umbrales_eficiente(df, columna='parrafo_anonimizado', umbral_bajo=0.85, umbral_alto=0.95, max_muestras=100):
    vectorizer = TfidfVectorizer()
    tfidf_matrix = vectorizer.fit_transform(df[columna].astype(str))
    
    n_filas = tfidf_matrix.shape[0]
    reporte = []
    encontrados = 0
    
    print(f"Analizando {n_filas} filas buscando similitudes > {umbral_bajo}...")

    # Procesamos fila por fila para no colapsar la RAM
    for i in range(n_filas):
        # Calculamos similitud de la fila i contra todas las posteriores j
        # Esto devuelve un array de (1, n_filas - i - 1)
        sim_vector = cosine_similarity(tfidf_matrix[i], tfidf_matrix[i+1:])[0]
        
        # Buscamos índices donde la similitud supere el umbral_bajo
        indices_match = np.where(sim_vector >= umbral_bajo)[0]
        
        for idx in indices_match:
            j = i + 1 + idx 
            sim_val = sim_vector[idx]
            
            reporte.append({
                'similitud': sim_val,
                'frase_1': df.iloc[i][columna],
                'frase_2': df.iloc[j][columna],
                'categoria': 'DUPLICADO' if sim_val > umbral_alto else 'DUDA'
            })
            encontrados += 1
            
        if encontrados >= max_muestras:
            print(f"Se han encontrado {encontrados} ejemplos. Deteniendo búsqueda.")
            break
            
    return pd.DataFrame(reporte)

df_prueba = analizar_umbrales_eficiente(df_parrafos, max_muestras=200)
df_prueba.to_csv('auditoria_similitud.csv', index=False)

Analizando 32499 filas buscando similitudes > 0.85...
Se han encontrado 327 ejemplos. Deteniendo búsqueda.


In [69]:
df_prueba[(df_prueba["categoria"]=="DUDA")&(df_prueba["similitud"]<0.99)].to_csv("duda.csv")

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def eliminar_duplicados_semanticos(df, columna='parrafo_anonimizado', umbral=0.95):
    """
    Identifica y elimina párrafos con una similitud superior al umbral (95%).
    """
    print(f"Filas originales: {len(df)}")

    # 1. Vectorización TF-IDF (convierte el texto a números según importancia de palabras)
    vectorizer = TfidfVectorizer()
    tfidf_matrix = vectorizer.fit_transform(df[columna].astype(str))

    # 2. Calcular la matriz de Similitud de Coseno (todos contra todos)
    # Esto genera una matriz de N x N donde cada celda es el % de similitud
    sim_matrix = cosine_similarity(tfidf_matrix)

    # 3. Lógica de filtrado
    # Usamos un set para guardar los índices que vamos a borrar
    a_eliminar = set()

    for i in range(len(sim_matrix)):
        # Si esta fila ya está marcada para borrar, la saltamos
        if i in a_eliminar:
            continue
        
        # Comparamos la fila 'i' con todas las siguientes 'j'
        for j in range(i + 1, len(sim_matrix)):
            if sim_matrix[i, j] > umbral:
                a_eliminar.add(j)

    df_limpio = df.drop(index=list(a_eliminar)).reset_index(drop=True)
    
    print(f"Filas eliminadas por redundancia (> {umbral*100}%): {len(a_eliminar)}")
    print(f"Filas resultantes: {len(df_limpio)}")
    
    return df_limpio

df_parrafos = eliminar_duplicados_semanticos(df_parrafos, umbral=0.95)
df_parrafos.to_csv('prospectos_unicos_semanticos_15.csv', index=False)

Filas originales: 32499
Filas eliminadas por redundancia (> 95.0%): 4720
Filas resultantes: 27779


### Clustering Semántico con Alta Resolución (K=15)
Se establece un valor de **$k=15$** basándose en la necesidad de aislar variables críticas de seguridad que en modelos de menor resolución quedaban diluidas. 

Este número de clústeres permite identificar dimensiones como:
* **Conducción y Maquinaria** (Cluster 6).
* **Salud Reproductiva y Embarazo** (Cluster 13).
* **Instrucciones Técnicas** de formatos específicos (Cluster 5).

In [72]:
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans

# Cargar modelo de embeddings (multilingüe para español)
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

# 1. Limpiar y vectorizar
textos = df_parrafos['parrafo_anonimizado'].tolist()
embeddings = model.encode(textos, show_progress_bar=True)

# 2. Clustering
num_clusters = 15 
clustering_model = KMeans(n_clusters=num_clusters, random_state=42)
df_parrafos['cluster'] = clustering_model.fit_predict(embeddings)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 380.16it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 869/869 [13:00<00:00,  1.11it/s]


In [4]:
df_parrafos.to_csv("prospectos_clusters15.csv", index=False)

### Ingeniería de Prompts: Abstracción Semántica
Para lograr una **Arquitectura de Información Universal** válida para cualquier fármaco, implementamos un prompt de "Abstracción Semántica". 

In [ ]:
def generar_prompt_final(cluster_id, ejemplos):
  ejemplos_str = "\n- ".join(ejemplos)
      

  prompt = f"""[INST] <<SYS>>
  Eres un PACIENTE que tiene una duda urgente y busca la respuesta en estos fragmentos. 
  TU REGLA MÁXIMA: Escribe ÚNICAMENTE la pregunta que harías a tu médico.
  PROHIBIDO usar palabras como: 'texto', 'fragmento', 'prospecto', 'diseño', 'JSON', 'información', 'sección', 'analizar'.
  PROHIBIDO mencionar nombres de medicamentos.
  <</SYS>>
  Sigue este razonamiento interno:
  1. Este fragmento "Puede disminuir la capacidad para conducir o manejar maquinaria por contener propilenglicol, ya que puede producir síntomas parecidos a los del alcohol." responde a la pregunta "¿Puedo conducir después de tomar [MEDICAMENTO]?".
  2. Este fragmento "Estas dosis se pueden repetir cada 4 horas." no responde a esa pregunta, responde a "¿Cada cuánto tiempo me tengo que tomar [MEDICAMENTO]?".
  3. Teniendo esto en cuenta, ¿qué PREGUNTA UNIVERSAL responden todos los fragmentos del grupo?
  Basándote en estos fragmentos:
  {ejemplos_str}

  Genera la PREGUNTA DIRECTA que un paciente normal (no experto) haría. 
  Ejemplos de estilo correcto: 
  - "¿Cuando fue la última fecha de revisión y dónde está el laboratorio?"
  - "¿Dónde debo guardar la medicina?"

  Responde exclusivamente en este formato JSON:
  {{
    "pregunta_del_paciente": "Escribe aquí la pregunta"
  }}
  [/INST]"""
  return prompt

## Conexión con Ollama

In [78]:
import requests
import json
import pandas as pd

def consultar_ollama_stream(prompt, modelo_id):
    url = "https://wiig.dia.fi.upm.es/ollama/v1/chat/completions"
    payload = {
        "model": modelo_id,
        "messages": [{"role": "user", "content": prompt}],
        "stream": True
    }
    
    texto_acumulado = ""
    try:
        with requests.post(url, json=payload, stream=True, timeout=300) as r:
            r.raise_for_status()
            for line in r.iter_lines():
                if not line: continue
                decoded = line.decode("utf-8").replace("data: ", "").strip()
                if decoded == "[DONE]": break
                try:
                    chunk = json.loads(decoded)
                    content = chunk["choices"][0].get("delta", {}).get("content", "")
                    texto_acumulado += content
                except: continue
        return texto_acumulado
    except Exception as e:
        return f"ERROR: {str(e)}"

### Evaluación del Comité LLM (Prompt Inicial)
Para interpretar el contenido semántico de los 15 clústeres generados por el algoritmo K-Means, creamos un comité de modelos de lenguaje locales ejecutados mediante Ollama. Utilizamos tres variantes para evaluar la estabilidad de la generación:
* **Llama3:latest:** Versión estándar (7B parámetros).
* **Llama3_hard:** Configurado con una temperatura baja (0.1) para forzar respuestas deterministas.
* **Llama3_medium:** Configurado con temperatura media (0.4) para permitir cierta creatividad estructural.

**Objetivo:** Solicitar a cada modelo que lea una muestra de párrafos de cada clúster y genere una pregunta que represente la duda del paciente. Se utiliza un *prompt* básico de extracción de intención.

In [76]:
anotadores = ["llama3:latest", "llama3_hard:latest", "llama3_medium:latest"]
resultados = []

In [80]:
import time

for c_id in sorted(df_parrafos['cluster'].unique()):
    fila = {}
    print(f" Procesando Cluster {c_id}...")
    
    # 1. Selección de variedad
    df_c = df_parrafos[df_parrafos['cluster'] == c_id]
    ejemplos_variados = df_c["parrafo_anonimizado"].to_list()
    
    # 2. Generación del prompt
    prompt_texto = generar_prompt_final(c_id, ejemplos_variados)
    
    fila = {"cluster": c_id, "ejemplos_usados": "|".join(ejemplos_variados)}
    
    # 3. Consulta a los LLMs
    for modelo in anotadores:
        print(f"  -> Consultando {modelo}...")
        respuesta = consultar_ollama_stream(prompt_texto, modelo)
        print(respuesta)
        fila[f"res_{modelo.replace(':', '_')}"] = respuesta
        time.sleep(1) # Pausa para no saturar el servidor

    resultados.append(fila)

df_final = pd.DataFrame(resultados)
df_final.to_csv("resultados_etiquetado_15topicos_nuevo.csv", index=False)

 Procesando Cluster 0...
  -> Consultando llama3:latest...
{
"pregunta_del_paciente": "¿Cuál es la dosis recomendada para adultos y niños?"
}
  -> Consultando llama3_hard:latest...
{
  "pregunta_del_paciente": "¿Cuál es la dosis recomendada para mí si peso xxx kg?"
}
  -> Consultando llama3_medium:latest...
{
  "pregunta_del_paciente": "¿Qué dosis debo tomar si tengo sobrepeso?"
}
 Procesando Cluster 1...
  -> Consultando llama3:latest...
{
"pregunta_del_paciente": "¿Puedo seguir tomando [MEDICAMENTO] si me voy a embarazar o estoy amamantando a un bebé?"
}
  -> Consultando llama3_hard:latest...
{
"pregunta_del_patient": "¿Es seguro tomar [MEDICAMENTO] si estoy embarazada ointentando quedarme embracada?"
  -> Consultando llama3_medium:latest...
{
"pregunta_del_paciente": "¿Puedo utilizar estos anticonceptivos hormonales combinados si estoy embarazada o intentando quedarme embarazada?"
}
 Procesando Cluster 2...
  -> Consultando llama3:latest...
{
"pregunta_del_paciente": "¿Qué debería h

In [ ]:
import pandas as pd
import json
import re

def extraer_pregunta_segura(texto):
    """
    Limpia y extrae la pregunta del JSON generado por el LLM, 
    gestionando texto basura y fallos de formato.
    """
    if pd.isna(texto) or "ERROR" in str(texto):
        return "Error en respuesta"

    match = re.search(r'\{.*\}', str(texto), re.DOTALL)
    
    if match:
        json_str = match.group()
        try:
            # Reemplazar comillas dobles repetidas
            json_str = json_str.replace('""', '"')
            datos = json.loads(json_str)
            
            pregunta = datos.get('pregunta_final')
            
            if pregunta:
                return pregunta.strip()
        except Exception:
            pass

    # 2. Si falla el JSON, buscamos patrones de texto
    # Busca frases que empiecen por "¿" y terminen por "?"
    preguntas_encontradas = re.findall(r'¿[^?]+\?', str(texto))
    if preguntas_encontradas:
        return preguntas_encontradas[-1].strip()

    return "No se pudo identificar la pregunta"


df_resultados = pd.read_csv('resultados_etiquetado_15topicos_nuevo.csv')

anotadores = [col for col in df_resultados.columns if col.startswith('res_')]

for col in anotadores:
    nombre_limpio = col.replace('res_', 'pregunta_')
    df_resultados[nombre_limpio] = df_resultados[col].apply(extraer_pregunta_segura)

columnas_formulario = ['cluster'] + [col for col in df_resultados.columns if col.startswith('pregunta_')]
print(df_resultados[columnas_formulario].head())

df_resultados[columnas_formulario].to_csv('preguntas_para_formulario_15_v3_similitud.csv', index=False)

   cluster                             pregunta_llama3_latest  \
0        0      ¿Qué hago si accidentalmente cargo una dosis?   
1        1  ¿Puedo seguir tomando este medicamento en el e...   
2        2  ¿Puedo usar [MEDICAMENTO] si tengo una deficie...   
3        3  ¿Puedo usar este medicamento con otros tratami...   
4        4  ¿Puedo desarrollar enfermedades graves durante...   

                         pregunta_llama3_hard_latest  \
0  ¿Cómo administro la dosis incorrecta accidenta...   
1  ¿Cómo funciona el método si me olvido tomar un...   
2  ¿Puedo usar [MEDICAMENTO] con pacientes con de...   
3  ¿Qué efectos secundarios graves espero que apa...   
4  ¿Qué síntomas graves puedo experimentar con es...   

                       pregunta_llama3_medium_latest  
0  ¿Cómo funcionan las cápsulas modificadas con l...  
1  ¿Puedo usar [MEDICAMENTO] durante el embarazo ...  
2  ¿Qué contiene [MEDICAMENTO] para ser peligroso...  
3  ¿Me protejo adecuadamente contra efectos secun...

### Auditoría de Calidad: Diagnóstico de Alucinaciones y Sesgo Léxico
La inspección manual de las salidas generadas por los tres modelos revela fallos críticos. Se han identificado cuatro tipos de error grave:

1. **Sobre-especificación y Ruido Contextual (Cluster 3 y 6):** El algoritmo de clustering ha agrupado párrafos por términos muy específicos. Como resultado, los modelos generan preguntas inservibles a nivel general, como *"¿Puedo llevar a mi hijo a tomar un baño...?"* (Cluster 6, Llama3_medium) o preguntan específicamente por cosméticos y psoriasis (Cluster 3).
2. **Inestabilidad del Formato y Lenguaje (Cluster 4 y 9):** Llama3_latest rompe el formato JSON y responde en inglés (*"What are the possible side effects..."*).
3. **Colapso de Contexto Administrativo (Cluster 11):** Al enfrentarse a fechas de revisión y firmas legales (información sin carga clínica), los LLMs fracasan estrepitosamente. Llama3_medium alucina con un incomprensible *"¿Qué es lo que esto es?"*.
4. **Falta de Consenso (Cluster 12):** Mientras *latest* pregunta por la lactancia, *hard* pregunta por la conducción y *medium* por la pediatría. El clúster es un "cajón de sastre" sin cohesión semántica.

**Decisión Metodológica:**
Esta evaluación demuestra empíricamente que la extracción directa ("resume este texto en una pregunta") falla cuando el texto base es altamente burocrático o hiper-específico. Para solucionar este colapso, probamos a cambiar los modelos utilizados y utilizar un promp más simple. 

### Experimento de Control: Evaluación de Modelos Avanzados con Prompt Simple
Para descartar que los fallos de sobre-especificación léxica y alucinaciones se debieran a las limitaciones cognitivas de los modelos pequeños (Llama3 8B), se diseña un experimento de control. 

**Metodología:**
Se despliega un comité de LLMs de gran escala y estado del arte (SOTA):
* **DeepSeek-R1 (32B)**
* **Llama-3.1 (70B)**
* **Gemma-3 (27B)**

Se utiliza un *prompt baseline* (simple y directo): *"Genera una pregunta breve que un paciente haría para obtener esta información"*. El objetivo es comprobar si la capacidad de abstracción de modelos de +27 billones de parámetros es suficiente para generalizar los clústeres sin necesidad de aplicar reglas de ingeniería de prompts.

In [ ]:
import requests
import json
import pandas as pd
import time


URL_BASE = "https://wiig.dia.fi.upm.es/ollama/v1/chat/completions"
COMITE_MODELOS = ["deepseek-r1:32b", "llama3.1:70b", "gemma3:27b"]

def consultar_modelo(modelo, prompt):
    """Realiza la petición al endpoint y limpia la respuesta."""
    payload = {
        "model": modelo,
        "messages": [
            {"role": "system", "content": "Eres un experto en lingüística médica. Responde exclusivamente con la pregunta solicitada, sin introducciones."},
            {"role": "user", "content": prompt}
        ],
        "temperature": 0.1
    }
    
    try:
        response = requests.post(URL_BASE, json=payload, timeout=300)
        response.raise_for_status()
        content = response.json()['choices'][0]['message']['content'].strip()
        
        if "</think>" in content:
            content = content.split("</think>")[-1].strip()
            
        return content
    except Exception as e:
        return f"ERROR: {str(e)}"

def procesar_experimento_consenso(df_clusters, columna_texto='parrafo_anonimizado'):
    """
    Agrupa por cluster, toma una muestra de párrafos representativos 
    y consulta al comité de modelos.
    """
    resultados_finales = []
    
    # Agrupamos por cluster ID
    clusters = df_clusters['cluster'].unique()
    
    for cluster_id in sorted(clusters):
        print(f"\n--- Procesando Cluster: {cluster_id} ---")
        
        # Tomamos los párrafos de este cluster (número limitado para no saturar)
        textos = df_clusters[df_clusters['cluster'] == cluster_id][columna_texto].head(10).tolist()
        contexto_cluster = " ".join(textos)
        
        prompt_gen = f"A partir de estos fragmentos de un prospecto médico, genera una pregunta breve que un paciente haría para obtener esta información: {contexto_cluster}"
        
        fila_resultado = {'cluster_id': cluster_id}
        
        for mod in COMITE_MODELOS:
            print(f"  > Consultando a {mod}...")
            pregunta = consultar_modelo(mod, prompt_gen)
            print(pregunta)
            fila_resultado[f'pregunta_{mod.split(":")[0]}'] = pregunta
            
        resultados_finales.append(fila_resultado)
        
    return pd.DataFrame(resultados_finales)

df_consenso = procesar_experimento_consenso(df_parrafos)

nombre_archivo = "consenso_preguntas_clusters_15top_modelos_nuevos.csv"
df_consenso.to_csv(nombre_archivo, index=False, encoding='utf-8')

print(df_consenso.head())


--- Procesando Cluster: 0 ---
  > Consultando a deepseek-r1:32b...
¿Cuánto sodio contiene este medicamento y cuál es la dosis recomendada?
  > Consultando a llama3.1:70b...
¿Cuál es la cantidad de sodio que contiene cada dosis del medicamento?
  > Consultando a gemma3:27b...
¿Cuántos comprimidos debo tomar al día y qué cantidad de sodio contienen?

--- Procesando Cluster: 1 ---
  > Consultando a deepseek-r1:32b...
¿Es seguro tomar este medicamento si estoy embarazada o en periodo de lactancia?
  > Consultando a llama3.1:70b...
¿Es seguro tomar este medicamento durante el embarazo o la lactancia?
  > Consultando a gemma3:27b...
¿Qué riesgos tiene este medicamento si estoy embarazada, planeo quedar embarazada o estoy amamantando?

--- Procesando Cluster: 2 ---
  > Consultando a deepseek-r1:32b...
¿Puedo tomar [MEDICAMENTO] con alimentos, bebidas o alcohol?
  > Consultando a llama3.1:70b...
¿Cuáles son los componentes del medicamento?
  > Consultando a gemma3:27b...
¿Qué ingredientes con

Las preguntas generadas revelan que, a pesar de la enorme potencia computacional de los modelos, el uso de un *prompt* simple genera resultados malos para una Arquitectura de Información Universal. Se han identificado tres fallos estructurales:

1. **Sesgo de Especificidad (Alucinación de Fármaco):** Al leer párrafos con ejemplos concretos, los modelos asumen que la pregunta debe incluir ese fármaco. Llama3.1 (70B) genera una pregunta exclusiva para el **VIH** en el Cluster 8, y Gemma-3 asume que el prospecto trata sobre **ácido acetilsalicílico** en el Cluster 13.
2. **Inestabilidad del Formato:** DeepSeek-R1, a pesar de las instrucciones del sistema, responde en inglés en el Cluster 6 (*"Who should not take [MEDICAMENTO]?"*).
3. **Limitaciones de Infraestructura (Timeouts):** Debido a la alta demanda computacional de los modelos de 70B y 32B, el entorno de ejecución (API UPM) presentó episodios de latencia severa y errores de lectura (Read Timeout en los clústeres 10 y 11). 

**Conclusión:**
Este experimento demuestra irrefutablemente que **el sesgo de especificidad viene de los datos (los prospectos), no del tamaño del LLM**.  Para evitar la especificidad de las preguntas, ahora que tenemos un conjunto de datos más reducido (con deduplicación semántica), probamos agrupaciones menores para buscar que sean menos especializadas.